## Setup

In [1]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [2]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [ ]:
# from ner import evaluation

In [3]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [4]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### wikiann

In [6]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann_sr = ner.ReadNERData()
wikiann_hr = ner.ReadNERData()

wikiann_sr_words, wikiann_sr_labels = wikiann_sr.read_dataset('wikiann', wikiann_label_map, lang='sr')
wikiann_hr_words, wikiann_hr_labels = wikiann_hr.read_dataset('wikiann', wikiann_label_map, lang='hr')

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

# Evaluate model

In [7]:
alignment = {
'B-organization': 'B-ORG',
'O': 'O',
'B-other': 'O',
'B-person': 'B-PER',
'I-person': 'I-PER',
'B-location': 'B-LOC',
'I-organization': 'I-ORG',
'I-other': 'O',
'I-location': 'I-LOC'
}

model_name = "tner/xlm-roberta-large-conll2003"
model_name_output = 'tner-xlm-roberta-large'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

tokenizer_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

In [ ]:
# model_evaluation.model.config.id2label

{0: 'B-LOC',
 1: 'B-MISC',
 2: 'B-ORG',
 3: 'I-LOC',
 4: 'I-MISC',
 5: 'I-ORG',
 6: 'I-PER',
 7: 'O'}

### wikiann - serbian

In [8]:
data_name = "wikiann_sr"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_sr_words, wikiann_sr_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [9]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.1121,0.1962,0.1427,3782
1,ORG,0.4879,0.3372,0.3988,3657
2,PER,0.7023,0.7801,0.7392,4139
3,micro,0.3786,0.4495,0.4110,11578
4,macro,0.4341,0.4378,0.4269,11578
5,weighted,0.4418,0.4495,0.4368,11578


In [10]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.4008,0.6888,0.5068,3782
1,B-ORG,0.6304,0.4145,0.5002,3657
2,B-PER,0.7929,0.8666,0.8281,4139
3,I-LOC,0.6763,0.1047,0.1813,9820
4,I-ORG,0.7628,0.2604,0.3883,7780
5,I-PER,0.9060,0.7162,0.8000,5962
6,O,0.7306,0.9835,0.8384,37048
7,accuracy,0.7130,72188,None,None
8,macro,0.7000,0.5764,0.5776,72188
9,weighted,0.7224,0.7130,0.6622,72188


### wikiann - croatian

In [12]:
data_name = "wikiann_hr"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_hr_words, wikiann_hr_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [13]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.6578,0.7647,0.7072,4862
1,ORG,0.6125,0.4217,0.4995,4100
2,PER,0.7781,0.8265,0.8016,4404
3,micro,0.6909,0.6799,0.6853,13366
4,macro,0.6828,0.6710,0.6694,13366
5,weighted,0.6835,0.6799,0.6746,13366


In [14]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.7409,0.8429,0.7886,4862
1,B-ORG,0.7258,0.4829,0.5800,4100
2,B-PER,0.8812,0.9178,0.8991,4404
3,I-LOC,0.6877,0.3165,0.4335,2818
4,I-ORG,0.8752,0.3493,0.4994,7285
5,I-PER,0.9441,0.7741,0.8507,5675
6,O,0.8670,0.9801,0.9201,57070
7,accuracy,0.8570,86214,None,None
8,macro,0.8174,0.6662,0.7102,86214
9,weighted,0.8538,0.8570,0.8394,86214
